# 01: Graphs on Shapes - Dual Representation (TopologicPy Native)

**Goal**: Implement the `GraphShape` class using pure TopologicPy primitives

**Learning Objectives**:
- Understand TopologicPy-native dual representation (Cluster + Graph)
- Create graphs that "live on" geometric shapes (Faces)
- Visualize topology and geometry using Topology.Show()
- Validate graph-shape consistency using TopologicPy methods

**Key Concept**: Every vertex in the TopologicPy Graph corresponds to a Face in the Cluster, and edges represent spatial adjacencies

## Setup

In [ ]:
# Add src to path
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'src'))

# TopologicPy imports
from topologicpy.Topology import Topology
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cluster import Cluster
from topologicpy.Graph import Graph

# Import TopologicPy helpers
from grammar.topologic_helpers import (
    rectangular_face,
    square_face,
    get_metadata,
    set_metadata,
    face_area,
    face_centroid,
    faces_adjacent,
    faces_overlap,
    faces_bounding_box,
    graph_from_faces_and_adjacencies,
    vertex_coordinates,
)

# Import graph-shape integration
from grammar.rules import GraphShape

print("✅ TopologicPy-native architecture loaded")
print("   - All geometry as Face objects")
print("   - All topology as Graph objects")
print("   - No conversion layer needed")

---

## Part 1: Understanding TopologicPy-Native GraphShape

The refactored `GraphShape` class maintains **pure TopologicPy dual representation**:
- **Geometry**: Cluster of Face objects (room polygons)
- **Topology**: Graph with Vertices at Face centroids, Edges for adjacencies

No custom classes, no networkx - TopologicPy all the way!

### 1.1: Creating Faces with Helper Functions

In [ ]:
# Create individual room faces using helper functions
# Layout: [A][B]
#         [C]   

face_a = rectangular_face(5, 4, origin=(0, 4), label='A')
face_b = rectangular_face(5, 4, origin=(5, 4), label='B')
face_c = rectangular_face(5, 4, origin=(0, 0), label='C')

print("Created TopologicPy Faces:")
print(f"  Face A: {type(face_a).__name__}, area = {face_area(face_a):.1f}")
print(f"  Face B: {type(face_b).__name__}, area = {face_area(face_b):.1f}")
print(f"  Face C: {type(face_c).__name__}, area = {face_area(face_c):.1f}")
print()

# Inspect metadata
print("Metadata attached to Face A:")
print(f"  label: {get_metadata(face_a, 'label')}")
print(f"  width: {get_metadata(face_a, 'width')}")
print(f"  height: {get_metadata(face_a, 'height')}")
print(f"  origin_x: {get_metadata(face_a, 'origin_x')}")
print(f"  origin_y: {get_metadata(face_a, 'origin_y')}")

### 1.2: Creating GraphShape from Faces

In [ ]:
# Create GraphShape using TopologicPy-native factory method
faces = [face_a, face_b, face_c]
adjacencies = [
    ('A', 'B'),  # Horizontal adjacency
    ('A', 'C')   # Vertical adjacency
]

gs = GraphShape.from_faces_and_adjacencies(faces, adjacencies)

print("GraphShape created:")
print(f"  Type: {type(gs).__name__}")
print(f"  Cluster type: {type(gs.cluster).__name__}")
print(f"  Graph type: {type(gs.graph).__name__}")
print(f"  Nodes: {gs.num_nodes()}")
print(f"  Edges: {gs.num_edges()}")
print(f"  Total area: {gs.total_area():.1f}")
print()

# Inspect individual faces
for face in gs.faces():
    label = get_metadata(face, 'label')
    width = get_metadata(face, 'width')
    height = get_metadata(face, 'height')
    print(f"  {label}: {width}×{height} = {face_area(face):.1f}")

★ Insight ─────────────────────────────────────

**TopologicPy-Native Architecture:**
- `rectangular_face()` creates TopologicPy Face with embedded Dictionary metadata
- `GraphShape.from_faces_and_adjacencies()` builds Cluster (geometry) + Graph (topology)
- Graph vertices positioned at Face centroids, edges connect adjacent rooms

**No conversion layer:** Faces stay as Faces throughout - TopologicPy algorithms work directly on geometry!

─────────────────────────────────────────────────

### 1.3: Graph Properties Using TopologicPy Methods

In [ ]:
# Analyze graph structure using TopologicPy Graph methods
vertices = gs.vertices()
edges = gs.edges()

print("Graph analysis:")
print(f"  Vertices: {len(vertices)}")
print(f"  Edges: {len(edges)}")
print()

# Node degrees using GraphShape helper methods
print("Node degrees:")
for face in gs.faces():
    label = get_metadata(face, 'label')
    degree = gs.degree(label)
    neighbors = gs.get_neighbors(label)
    print(f"  {label}: degree={degree}, neighbors={neighbors}")

---

## Part 2: Validation - Ensuring Consistency

Using TopologicPy's geometric methods for validation:
1. **No overlaps**: `Topology.Intersect()` for overlap detection
2. **Edges match adjacencies**: `Topology.SharedEdges()` for adjacency
3. **Connectivity**: Graph structure analysis

### 2.1: Overlap Detection

In [ ]:
# Check for overlapping shapes using TopologicPy methods
overlaps = gs.find_overlaps()

if overlaps:
    print(f"⚠️  Found {len(overlaps)} overlaps:")
    for n1, n2 in overlaps:
        print(f"  {n1} ↔ {n2}")
else:
    print("✅ No overlapping shapes detected")

### 2.2: Adjacency Validation

In [ ]:
# Verify edges correspond to geometric adjacencies
print("Edge validation using TopologicPy.SharedEdges():")
for edge in gs.edges():
    edge_verts = Edge.Vertices(edge)
    if len(edge_verts) == 2:
        label1 = get_metadata(edge_verts[0], 'label')
        label2 = get_metadata(edge_verts[1], 'label')
        
        face1 = gs.get_face_by_label(label1)
        face2 = gs.get_face_by_label(label2)
        
        if face1 and face2:
            adjacent = faces_adjacent(face1, face2)
            status = "✅" if adjacent else "❌"
            print(f"  {status} ({label1}, {label2}): geometrically adjacent = {adjacent}")

# Find missing adjacencies
missing_edges = gs.find_missing_adjacencies()
if missing_edges:
    print(f"\n⚠️  Found {len(missing_edges)} geometric adjacencies not in graph:")
    for n1, n2 in missing_edges:
        print(f"  {n1} ↔ {n2}")
else:
    print("\n✅ All geometric adjacencies represented in graph")

### 2.3: Comprehensive Validation

In [ ]:
# Run all validation checks
is_valid, issues = gs.validate()

if is_valid:
    print("✅ GraphShape is valid")
else:
    print(f"❌ GraphShape has {len(issues)} issue(s):")
    for issue in issues:
        print(f"  - {issue}")

★ Insight ─────────────────────────────────────

**TopologicPy Geometric Validation:**
- `faces_adjacent()` uses `Topology.SharedEdges()` - detects shared boundaries
- `faces_overlap()` uses `Topology.Intersect()` - finds interior overlaps
- No manual distance calculations needed - library's proven algorithms!

─────────────────────────────────────────────────

---

## Part 3: Visualizing with Topology.Show()

**New approach**: Use `Topology.Show()` to display both geometry (Cluster) and topology (Graph) together!

In [ ]:
# Visualize GraphShape using native TopologicPy visualization
print("🎨 Displaying dual representation with Topology.Show()...")
print("   - Blue faces: room geometries")
print("   - Graph edges: adjacency connections")

Topology.Show(
    gs.cluster,
    gs.graph,
    renderer="notebook",
    sagitta=0.05,
    absolute=False,
    faceOpacity=0.5,
    backgroundColor="white",
    width=800,
    height=600
)

★ Insight ─────────────────────────────────────

**Topology.Show() Dual Visualization:**
- Passes BOTH `cluster` (Faces) and `graph` (topology) to single call
- TopologicPy overlays graph edges on geometry automatically
- No separate pyvis/networkx needed - integrated visualization!

─────────────────────────────────────────────────

---

## Part 4: Building Complex Layouts with Factory Methods

GraphShape provides factory methods that create common layouts directly as TopologicPy primitives

### 4.1: Grid Layout

In [ ]:
# Create 2x3 grid using factory method
gs_grid = GraphShape.from_grid(
    base_width=20,
    base_height=15,
    rows=2,
    cols=3,
    origin=(0, 0)
)

print("Grid GraphShape:")
print(f"  Nodes: {gs_grid.num_nodes()}")
print(f"  Edges: {gs_grid.num_edges()}")
print(f"  Total area: {gs_grid.total_area():.1f}")
print(f"  Bounding box: {gs_grid.bounding_box()}")
print()

# Show grid topology
print("Grid cells:")
for face in gs_grid.faces():
    label = get_metadata(face, 'label')
    row = get_metadata(face, 'row')
    col = get_metadata(face, 'col')
    neighbors = gs_grid.get_neighbors(label)
    print(f"  {label} (row={row}, col={col}): {len(neighbors)} neighbors")

In [ ]:
# Visualize grid
print("🎨 Grid layout with mesh topology:")

Topology.Show(
    gs_grid.cluster,
    gs_grid.graph,
    renderer="notebook",
    sagitta=0.05,
    faceOpacity=0.4,
    backgroundColor="white",
    width=900,
    height=600
)

### 4.2: Horizontal Split Layout

In [ ]:
# Create linear apartment layout
gs_linear = GraphShape.from_horizontal_split(
    base_width=30,
    base_height=10,
    ratios=[0.2, 0.3, 0.3, 0.2],
    labels=['Entrance', 'Kitchen', 'Living', 'Bedroom'],
    origin=(0, 0)
)

print("Linear apartment:")
print(f"  Nodes: {gs_linear.num_nodes()}")
print(f"  Edges: {gs_linear.num_edges()}")
print(f"  Total area: {gs_linear.total_area():.1f}")
print()

# Inspect rooms
for face in gs_linear.faces():
    label = get_metadata(face, 'label')
    width = get_metadata(face, 'width')
    degree = gs_linear.degree(label)
    print(f"  {label}: width={width:.1f}m, degree={degree}")

In [ ]:
# Visualize linear layout
print("🎨 Linear apartment with chain topology:")

Topology.Show(
    gs_linear.cluster,
    gs_linear.graph,
    renderer="notebook",
    sagitta=0.05,
    faceOpacity=0.5,
    backgroundColor="white",
    width=1000,
    height=400
)

---

## Part 5: Complex Apartment Layout

Create a realistic apartment with manually positioned rooms

In [ ]:
# Create realistic apartment layout with TopologicPy faces
complex_faces = [
    rectangular_face(3, 3, origin=(0, 4), label='Entrance'),
    rectangular_face(10, 2, origin=(3, 5), label='Corridor'),
    rectangular_face(4, 4, origin=(3, 8), label='Kitchen'),
    rectangular_face(6, 5, origin=(7, 8), label='Living'),
    rectangular_face(3, 3, origin=(3, 1), label='Bathroom'),
    rectangular_face(5, 4, origin=(6, 0), label='Bedroom1'),
    rectangular_face(4, 3, origin=(11, 0), label='Bedroom2')
]

complex_adjacencies = [
    ('Entrance', 'Corridor'),
    ('Corridor', 'Kitchen'),
    ('Corridor', 'Living'),
    ('Corridor', 'Bathroom'),
    ('Corridor', 'Bedroom1'),
    ('Kitchen', 'Living'),
    ('Bedroom1', 'Bedroom2')
]

complex_gs = GraphShape.from_faces_and_adjacencies(complex_faces, complex_adjacencies)

print("Complex apartment:")
print(f"  Rooms: {complex_gs.num_nodes()}")
print(f"  Connections: {complex_gs.num_edges()}")
print(f"  Total area: {complex_gs.total_area():.1f} m²")
print(f"  Bounding box: {complex_gs.bounding_box()}")
print()

# Find central room (highest degree)
max_degree = 0
central_room = None
for face in complex_gs.faces():
    label = get_metadata(face, 'label')
    degree = complex_gs.degree(label)
    if degree > max_degree:
        max_degree = degree
        central_room = label

print(f"Central room: {central_room} (degree {max_degree})")
print(f"Connected to: {complex_gs.get_neighbors(central_room)}")

In [ ]:
# Visualize complex apartment
print("🎨 Complex apartment with hub topology:")
print("   Corridor acts as central hub connecting all rooms")

Topology.Show(
    complex_gs.cluster,
    complex_gs.graph,
    renderer="notebook",
    sagitta=0.05,
    faceOpacity=0.5,
    backgroundColor="white",
    width=1000,
    height=800
)

# Export to HTML for external viewing
Path("viz_outputs").mkdir(exist_ok=True)
Topology.ExportToHTML(
    complex_gs.cluster,
    path="viz_outputs/01_complex_apartment_geometry.html",
    title="Complex Apartment - TopologicPy Native"
)
print("\n✅ Exported to: viz_outputs/01_complex_apartment_geometry.html")

★ Insight ─────────────────────────────────────

**Factory Methods vs Manual Construction:**
- `from_grid()` and `from_horizontal_split()` create common patterns automatically
- Manual construction (complex apartment) gives full control over positions
- Both produce identical GraphShape structure: Cluster + Graph

**Architecture advantage:** All methods work with same TopologicPy primitives - no special cases!

─────────────────────────────────────────────────

---

## Part 6: Tests - GraphShape Validation

In [ ]:
print("="*70)
print("GRAPHSHAPE TEST SUITE (TopologicPy Native)")
print("="*70)
print()

tests_passed = 0
tests_total = 0

# Test 1: Area conservation in grid
tests_total += 1
gs_test = GraphShape.from_grid(100, 100, rows=4, cols=4)
expected_area = 100 * 100
actual_area = gs_test.total_area()
if abs(actual_area - expected_area) < 0.01:
    print(f"✅ Test 1: Area conservation ({actual_area:.1f} ≈ {expected_area:.1f})")
    tests_passed += 1
else:
    print(f"❌ Test 1: Area mismatch ({actual_area:.1f} vs {expected_area:.1f})")

# Test 2: No overlaps in grid
tests_total += 1
overlaps = gs_test.find_overlaps()
if len(overlaps) == 0:
    print("✅ Test 2: Grid has no overlapping faces")
    tests_passed += 1
else:
    print(f"❌ Test 2: Found {len(overlaps)} overlaps")

# Test 3: Graph has correct node count
tests_total += 1
expected_nodes = 4 * 4
actual_nodes = gs_test.num_nodes()
if actual_nodes == expected_nodes:
    print(f"✅ Test 3: Correct node count ({actual_nodes})")
    tests_passed += 1
else:
    print(f"❌ Test 3: Node count mismatch ({actual_nodes} vs {expected_nodes})")

# Test 4: All graph edges have geometric adjacency
tests_total += 1
valid_adjacencies = True
for edge in gs_test.edges():
    edge_verts = Edge.Vertices(edge)
    if len(edge_verts) == 2:
        label1 = get_metadata(edge_verts[0], 'label')
        label2 = get_metadata(edge_verts[1], 'label')
        face1 = gs_test.get_face_by_label(label1)
        face2 = gs_test.get_face_by_label(label2)
        if face1 and face2:
            if not faces_adjacent(face1, face2):
                valid_adjacencies = False
                break

if valid_adjacencies:
    print("✅ Test 4: All edges correspond to geometric adjacencies")
    tests_passed += 1
else:
    print("❌ Test 4: Some edges don't match adjacencies")

# Test 5: Linear split creates chain topology
tests_total += 1
gs_chain = GraphShape.from_horizontal_split(30, 10, [0.33, 0.34, 0.33])
# Chain graph has exactly 2 endpoints (degree 1)
endpoint_count = 0
for face in gs_chain.faces():
    label = get_metadata(face, 'label')
    if gs_chain.degree(label) == 1:
        endpoint_count += 1

if endpoint_count == 2:
    print(f"✅ Test 5: Linear split creates chain topology (2 endpoints)")
    tests_passed += 1
else:
    print(f"❌ Test 5: Expected 2 endpoints, got {endpoint_count}")

# Test 6: Bounding box calculation
tests_total += 1
bbox = gs_test.bounding_box()
expected_bbox = (0, 0, 100, 100)
if bbox == expected_bbox:
    print(f"✅ Test 6: Bounding box correct: {bbox}")
    tests_passed += 1
else:
    print(f"❌ Test 6: Bounding box mismatch: {bbox} vs {expected_bbox}")

# Test 7: Validation passes for valid GraphShape
tests_total += 1
is_valid, issues = complex_gs.validate()
if is_valid:
    print("✅ Test 7: Complex apartment validates successfully")
    tests_passed += 1
else:
    print(f"❌ Test 7: Validation failed with {len(issues)} issues")

# Test 8: All faces are TopologicPy Face objects
tests_total += 1
all_faces_valid = all("Face" in str(type(f)) for f in complex_gs.faces())
if all_faces_valid:
    print("✅ Test 8: All shapes are TopologicPy Face objects")
    tests_passed += 1
else:
    print("❌ Test 8: Some shapes are not Face objects")

# Test 9: Graph is TopologicPy Graph object
tests_total += 1
if "Graph" in str(type(complex_gs.graph)):
    print("✅ Test 9: Topology is TopologicPy Graph object")
    tests_passed += 1
else:
    print("❌ Test 9: Topology is not Graph object")

print()
print("="*70)
print(f"RESULTS: {tests_passed}/{tests_total} tests passed")
print("="*70)

if tests_passed == tests_total:
    print("🎉 ALL TESTS PASSED!")
    print("\n✅ TopologicPy-native architecture verified:")
    print("   - All geometry as Face objects")
    print("   - All topology as Graph objects")
    print("   - No conversion layer needed")
    print("   - Validation uses TopologicPy methods")
else:
    print(f"⚠️  {tests_total - tests_passed} test(s) failed")

---

## Summary

**What we learned**:
1. ✅ `GraphShape` uses pure TopologicPy: Cluster (Faces) + Graph (topology)
2. ✅ Helper functions (`rectangular_face()`) create Faces with embedded metadata
3. ✅ Validation uses TopologicPy geometric methods (`SharedEdges`, `Intersect`)
4. ✅ Factory methods create common layouts (grid, linear) as TopologicPy primitives
5. ✅ `Topology.Show()` displays both geometry and topology in single visualization
6. ✅ No networkx, no pyvis, no conversion layer - TopologicPy all the way!

**Architecture Benefits**:
- **Single source of truth**: Faces stay as Faces throughout
- **Proven algorithms**: TopologicPy's geometric methods for adjacency/overlap
- **Simpler codebase**: 700+ lines of custom classes → 300 lines of helpers
- **Integrated visualization**: Topology.Show() handles dual display

**Next steps**:
- `02_Transformation_Rules.ipynb` - Implement graph grammar rules using TopologicPy Face subdivision
- Add rule-based transformations (split, merge) that operate on Faces directly
- Explore TopologicPy's advanced geometric operations for complex transformations